# Cab Price 

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score,mean_absolute_error

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score,precision_score,recall_score, f1_score,classification_report


from joblib import load, dump


# 
- --

# EDA and Data Processing

## Load Data set

In [11]:
df = pd.read_csv(r'../Data/UrbanCabFare.csv')


## Basic Data Exploration

In [12]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   User_ID               1000 non-null   str    
 1   Driver_Name           985 non-null    str    
 2   City                  1000 non-null   str    
 3   Ride_Date             1000 non-null   str    
 4   Hour_of_Day           1000 non-null   int64  
 5   Day_of_Week           1000 non-null   str    
 6   Distance_km           1000 non-null   float64
 7   Traffic_Level         1000 non-null   str    
 8   Type_of_vehicle       1000 non-null   str    
 9   No_of_active_drivers  1000 non-null   int64  
 10  Ride_Requests         1000 non-null   int64  
 11  Demand_Supply_Ratio   1000 non-null   float64
 12  Trip_Duration         1000 non-null   int64  
 13  Base_Fare             1000 non-null   float64
 14  Surge_Multiplier      1000 non-null   float64
 15  Final_Fare            1000 non-nu

In [13]:
df['Driver_Name'] = df['Driver_Name'].fillna('Unknown')


In [14]:
df.dtypes


User_ID                     str
Driver_Name                 str
City                        str
Ride_Date                   str
Hour_of_Day               int64
Day_of_Week                 str
Distance_km             float64
Traffic_Level               str
Type_of_vehicle             str
No_of_active_drivers      int64
Ride_Requests             int64
Demand_Supply_Ratio     float64
Trip_Duration             int64
Base_Fare               float64
Surge_Multiplier        float64
Final_Fare              float64
dtype: object

In [15]:
cat_cols = ['User_ID','Driver_Name','City','Traffic_Level','Type_of_vehicle']
# date_time_cols = ['Ride_Date','Hour_of_Day','Day_of_Week']

for col in cat_cols:
    df[col] = df[col].astype('category')
df['Ride_Date'] = pd.to_datetime(df['Ride_Date'],format="%d-%m-%Y")



In [16]:
df.describe(include=["int","float"])    


,Hour_of_Day,Distance_km,No_of_active_drivers,Ride_Requests,Demand_Supply_Ratio,Trip_Duration,Base_Fare,Surge_Multiplier,Final_Fare
count,1000.000000,1000.000000,1000.00000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,11.651000,17.911070,52.52000,108.047000,3.569190,55.165000,257.477840,1.436200,370.238310
std,7.020576,9.634464,27.47334,55.394199,4.734492,27.946857,117.993113,0.343435,199.232597
min,0.000000,1.010000,5.00000,10.000000,0.100000,6.000000,27.120000,1.000000,32.540000
25%,5.750000,9.567500,28.00000,60.000000,1.157500,32.000000,159.650000,1.000000,208.205000
50%,12.000000,18.180000,54.00000,110.500000,2.040000,54.000000,256.740000,1.500000,338.580000
75%,18.000000,25.897500,76.00000,156.000000,4.022500,76.250000,352.230000,1.700000,493.767500
max,23.000000,34.990000,100.00000,200.000000,39.400000,122.000000,498.200000,2.000000,939.760000


# 
- --

# Fare Model Preparation (Regression)

## Data for Fare

In [17]:
X_fare_df = df[[
    "City",
    "Type_of_vehicle",
    "Distance_km",
    "Trip_Duration",
    "Surge_Multiplier"
]]

y_fare_df =df["Final_Fare"]


## Splitingting the data

In [18]:
X_fare_train,X_fare_test,y_fare_train,y_fare_test = train_test_split(X_fare_df,y_fare_df,test_size=0.2,random_state =42)



## Building The Pipeline 

### Fare Model

In [19]:
numerical_features_fare = [
    "Distance_km",
    "Trip_Duration",
    "Surge_Multiplier"
]
categorical_features = [
    "City",
    "Type_of_vehicle"
]

preprocessor_fare = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "num",
            "passthrough",
            numerical_features_fare
        )
    ]
)


In [20]:
fare_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor_fare),
        ("regressor", LinearRegression())
    ]
)


## Train The Model

### Fare Model

In [21]:
fare_model.fit(X_fare_train,y_fare_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](5,)","['City','Type_of_vehicle','Distance_km','Trip_Duration','Surge_Multiplier']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,5
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of c

## Generate Prediction

### Fare Prediction

In [22]:
y_fare_pred = fare_model.predict(X_fare_test)


#### Fare Score 

In [23]:
fare_mae = mean_absolute_error(
    y_fare_test,
    y_fare_pred
)

fare_rmse = np.sqrt(
    mean_squared_error(
        y_fare_test,
        y_fare_pred
    )
)

fare_r2 = r2_score(
    y_fare_test,
    y_fare_pred
)


In [24]:
print("\n===== FARE PREDICTION MODEL =====")
print(f"MAE  : ₹{fare_mae:.2f}")
print(f"RMSE : ₹{fare_rmse:.2f}")
print(f"R²   : {fare_r2:.4f}")



===== FARE PREDICTION MODEL =====
MAE  : ₹30.27
RMSE : ₹40.17
R²   : 0.9570


In [ ]:
comparison = pd.DataFrame({
    "Actual_Fare": y_fare_test.values,
    "Predicted_Fare": y_fare_pred
})

comparison["Error"] = (
    comparison["Actual_Fare"] -
    comparison["Predicted_Fare"]
)

comparison.head(20)


,Actual_Fare,Predicted_Fare,Error
0,421.04,495.698042,-74.658042
1,137.18,222.277445,-85.097445
2,525.77,514.519484,11.250516
3,377.84,435.904742,-58.064742
4,222.67,200.433438,22.236562
5,542.69,596.056285,-53.366285
6,647.44,617.646211,29.793789
7,429.36,510.996679,-81.636679
8,448.04,535.061584,-87.021584
9,507.04,511.487176,-4.447176


In [ ]:
comparison.describe()


,Actual_Fare,Predicted_Fare,Error
count,200.000000,200.000000,200.000000
mean,387.697000,389.908043,-2.211043
std,194.305841,184.895263,40.211410
min,78.760000,-6.177283,-102.265344
25%,238.357500,255.331309,-22.924298
50%,354.005000,378.286846,1.052658
75%,494.475000,514.020878,22.050758
max,929.940000,825.648397,104.291603


In [ ]:
print(
    df[
        [
            "Distance_km",
            "Trip_Duration",
            "Surge_Multiplier",
            "Final_Fare"
        ]
    ].corr()["Final_Fare"].sort_values(ascending=False)
)


Final_Fare          1.000000
Distance_km         0.854009
Trip_Duration       0.819275
Surge_Multiplier    0.456232
Name: Final_Fare, dtype: float64


In [ ]:


mae = mean_absolute_error(
    comparison["Actual_Fare"],
    comparison["Predicted_Fare"]
)

rmse = np.sqrt(
    mean_squared_error(
        comparison["Actual_Fare"],
        comparison["Predicted_Fare"]
    )
)

r2 = r2_score(
    comparison["Actual_Fare"],
    comparison["Predicted_Fare"]
)

print(f"MAE  : ₹{mae:.2f}")
print(f"RMSE : ₹{rmse:.2f}")
print(f"R²   : {r2:.4f}")


MAE  : ₹30.27
RMSE : ₹40.17
R²   : 0.9570


## Exporting model file

In [30]:
destination = "../ML_models/linear_reg_fare_model.pkl"

dump(fare_model, destination)


['../ML_models/linear_reg_fare_model.pkl']

#
- --

In [ ]:
# Get the trained Linear Regression model
regressor = fare_model.named_steps["regressor"]

# Get the preprocessor
preprocessor = fare_model.named_steps["preprocessor"]

# Get feature names after preprocessing
feature_names = preprocessor.get_feature_names_out()

# Get coefficients
coefficients = regressor.coef_

# Create a readable table
coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
})

coef_df


,Feature,Coefficient
0,cat__City_Bengaluru,1.974498
1,cat__City_Chennai,5.111320
2,cat__City_Delhi,0.126309
3,cat__City_Hyderabad,-2.886349
4,cat__City_Mumbai,-3.357922
5,cat__City_Pune,-0.967856
6,cat__Type_of_vehicle_Auto,-20.692702
7,cat__Type_of_vehicle_Bike,-39.420574
8,cat__Type_of_vehicle_Hatchback,-7.840209
9,cat__Type_of_vehicle_SUV,56.863765


In [ ]:
pd.set_option("display.max_rows", None)

coef_df


,Feature,Coefficient
0,cat__City_Bengaluru,1.974498
1,cat__City_Chennai,5.111320
2,cat__City_Delhi,0.126309
3,cat__City_Hyderabad,-2.886349
4,cat__City_Mumbai,-3.357922
5,cat__City_Pune,-0.967856
6,cat__Type_of_vehicle_Auto,-20.692702
7,cat__Type_of_vehicle_Bike,-39.420574
8,cat__Type_of_vehicle_Hatchback,-7.840209
9,cat__Type_of_vehicle_SUV,56.863765


In [ ]:
print("Distance:")
print("Min:", df["Distance_km"].min())
print("Max:", df["Distance_km"].max())

print("\nTrip Duration:")
print("Min:", df["Trip_Duration"].min())
print("Max:", df["Trip_Duration"].max())

print("\nSurge:")
print("Min:", df["Surge_Multiplier"].min())
print("Max:", df["Surge_Multiplier"].max())


Distance:
Min: 1.01
Max: 34.99

Trip Duration:
Min: 6
Max: 122

Surge:
Min: 1.0
Max: 2.0


In [ ]:
test_input = pd.DataFrame({
    "City": ["Mumbai"],
    "Type_of_vehicle": ["SUV"],
    "Distance_km": [10],
    "Trip_Duration": [60],
    "Surge_Multiplier": [1.5]
})

fare_model.predict(test_input)


array([307.75718543])